# Wanda — AI Data Engineer for Microsoft Fabric

Investigate a failed Microsoft Fabric pipeline — or audit one before it runs — and get an evidence-backed root-cause report, right here in this notebook.

**Three steps:** install → configure → run.

## 1. Install

In [ ]:
%pip install "wanda-fabric[sql]"

# The [sql] extra also needs the OS-level Microsoft ODBC Driver 18 for SQL Server.
# If the SQL checks come back unavailable, see docs/GETTING_STARTED.md.

## 2. Configure

Wanda needs access to **your Fabric workspace** and an **LLM key**.

- **LLM:** an Anthropic API key (console.anthropic.com) — ~cents per investigation.
- **Fabric:** a Service Principal with read access to the workspace. *(A future version will use the notebook's own workspace identity, removing this step.)*

> First time? **[docs/GETTING_STARTED.md](../docs/GETTING_STARTED.md)** walks through creating the Service Principal and getting each value below.
>
> Prefer Fabric **workspace secrets** / Key Vault over pasting keys inline.

In [ ]:
import os

# --- LLM (bring your own key) ---
os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

# --- Microsoft Fabric (Service Principal) ---
os.environ["FABRIC_TENANT_ID"]     = "your-tenant-guid"
os.environ["FABRIC_CLIENT_ID"]     = "your-sp-client-id"
os.environ["FABRIC_CLIENT_SECRET"] = "your-sp-secret"
os.environ["FABRIC_WORKSPACE_ID"]  = "your-workspace-guid"

## 3. Investigate a failed pipeline

In [ ]:
from wanda import Wanda

wanda = Wanda()
report = wanda.investigate("Your Failed Pipeline Name")   # ← a real failed pipeline in YOUR workspace
report.display()

## (Optional) Pre-run scan — catch failures *before* the pipeline runs

Audits every activity against the live workspace and reports what will fail.

In [ ]:
wanda.scan("Your Pipeline Name").display()

---
### What Wanda sends to the LLM

Notebook source, pipeline structure, and table/column names — so the model can reason about the failure. During a pre-run **scan**, Wanda may also read a *small sample* of rows (e.g. `SELECT TOP 1 *`) to validate data — never bulk data, and never your secrets. Wanda is read-only (SQL is restricted to `SELECT`/`WITH` in code) and never modifies your workspace.

New to the setup? See **[docs/GETTING_STARTED.md](../docs/GETTING_STARTED.md)**.

### Tell us how it went
- **Form** (no setup needed): *[link — coming with your invite]*
- **From here**, if you've opted into telemetry (`WANDA_TELEMETRY=on`):

  ```python
  report.feedback(True, "nailed the missing-table root cause")   # or False, "missed it"
  ```

  Sends an anonymous 👍/👎 plus your note — never your data or secrets.